# Distilled Tiny RNNs — longer training and repeated seeds

**Purpose:** check training convergence and the stability of the first pilot's descriptive results. This remains a synthetic, conditional first-stage experiment with a parametric rare group. It does not establish recovery of the generating mechanism or reproduce the published method.

Open this notebook in a fresh Colab session and select a **T4 GPU**. The first cell downloads a fixed public revision without a GitHub token and mounts your Google Drive for checkpoints. Run the cells in order.

The follow-up allows 500 teacher steps and 1,000 steps per student arm. A phase can pause after 10 minutes; each study invocation has a soft 30-minute training budget, checked between steps/phases. Setup, saving, and evaluation can add time. Rerun the same notebook and run name to resume. These limits do not guarantee convergence or a particular GPU runtime. Disconnect the runtime when finished.


In [ ]:
"""Download one pinned public GitHub revision for a Colab launcher.

This standard-library module is embedded into the launchers. Each invocation
installs into a fresh runtime-local destination; saved runs belong elsewhere.
"""
from pathlib import Path, PurePosixPath
import io
import os
import re
import shutil
import stat
import tempfile
import urllib.error
import urllib.parse
import urllib.request
import zipfile


_ARCHIVE_LIMIT = 50 * 1024 * 1024
_EXTRACTED_LIMIT = 100 * 1024 * 1024


class _GitHubRedirectHandler(urllib.request.HTTPRedirectHandler):
    def redirect_request(self, req, fp, code, msg, headers, newurl):
        parsed = urllib.parse.urlsplit(newurl)
        if (parsed.scheme != "https" or parsed.hostname != "codeload.github.com"
                or parsed.username is not None or parsed.password is not None
                or parsed.port not in (None, 443)):
            raise RuntimeError("GitHub returned an unexpected download destination")
        return super().redirect_request(req, fp, code, msg, headers, newurl)


def _extract_project_archive(payload, destination):
    """Validate all archive paths before extracting a single GitHub root."""
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if os.path.lexists(destination):
        raise ValueError("Use a fresh empty destination path for the project")
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        entries = archive.infolist()
        if not entries or len(entries) > 10000:
            raise ValueError("Invalid project archive entry count")
        roots, seen, planned = set(), set(), []
        total_size = 0
        for entry in entries:
            name = entry.filename
            components = name.rstrip("/").split("/")
            mode = stat.S_IFMT(entry.external_attr >> 16)
            if (name.startswith("/") or "\\" in name or any(p in ("", ".", "..") for p in components)
                    or mode not in (0, stat.S_IFREG, stat.S_IFDIR) or entry.flag_bits & 1):
                raise ValueError("Unsafe project archive entry")
            roots.add(components[0])
            relative = PurePosixPath(*components[1:])
            normalized = str(relative)
            if normalized in seen:
                raise ValueError("Duplicate project archive entry")
            seen.add(normalized)
            if len(components) == 1:
                if not entry.is_dir():
                    raise ValueError("Project archive must contain a root directory")
                continue
            total_size += entry.file_size
            if total_size > _EXTRACTED_LIMIT:
                raise ValueError("Project archive is too large")
            planned.append((entry, relative))
        if len(roots) != 1 or not planned:
            raise ValueError("Project archive must have one root directory")
        files = {str(path) for entry, path in planned if not entry.is_dir()}
        for _, path in planned:
            if any(str(parent) in files for parent in path.parents):
                raise ValueError("Conflicting project archive paths")
        staging = Path(tempfile.mkdtemp(prefix=".colab-project-", dir=destination.parent))
        try:
            for entry, relative in planned:
                target = staging.joinpath(*relative.parts)
                if entry.is_dir():
                    target.mkdir(parents=True, exist_ok=True)
                else:
                    target.parent.mkdir(parents=True, exist_ok=True)
                    with archive.open(entry) as source, target.open("xb") as output:
                        shutil.copyfileobj(source, output, length=1024 * 1024)
            # Caller selects an unused path. Recheck to avoid replacing a path
            # that appeared while downloading/extracting.
            if os.path.lexists(destination):
                raise ValueError("Project destination already exists")
            staging.rename(destination)
        finally:
            if staging.exists():
                shutil.rmtree(staging)
    return destination


def fetch_public_project(repo, commit, destination):
    """Fetch and safely extract a public ZIP pinned to a full Git commit."""
    if (not isinstance(repo, str)
            or not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9-]{0,38}/[A-Za-z0-9_.-]{1,100}", repo)
            or repo.rsplit("/", 1)[-1] in {".", ".."}):
        raise ValueError("Repository must be an owner/name pair")
    if not isinstance(commit, str) or not re.fullmatch(r"[0-9a-fA-F]{40}", commit):
        raise ValueError("A full 40-character Git commit is required")
    if os.path.lexists(destination):
        raise ValueError("Use a fresh destination path for the project")
    request = urllib.request.Request(
        f"https://github.com/{repo}/archive/{commit}.zip",
        headers={"User-Agent": "distilled-rnns-colab"},
    )
    try:
        opener = urllib.request.build_opener(_GitHubRedirectHandler())
        with opener.open(request, timeout=60) as response:
            payload = response.read(_ARCHIVE_LIMIT + 1)
    except urllib.error.HTTPError as error:
        if error.code == 404:
            raise RuntimeError(
                "GitHub could not find the pinned revision. Check that the repository is public."
            ) from None
        raise RuntimeError(f"GitHub project download failed (HTTP {error.code}).") from None
    except (OSError, RuntimeError, ValueError):
        raise RuntimeError("GitHub project download failed. Check your connection and try again.") from None
    if len(payload) > _ARCHIVE_LIMIT:
        raise RuntimeError("The project download exceeds the size limit")
    try:
        return _extract_project_archive(payload, destination)
    except Exception:
        raise RuntimeError("The downloaded project archive could not be safely installed") from None


from pathlib import Path
import os, sys, hashlib, json, subprocess, tempfile
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

EXPECTED_SOURCE_SHA256 = "1b4d8e6c58188e9e16b6526a0f24bd9ec2ef20ddb0021d54a433db3e778b8255"
GITHUB_REPOSITORY = "WeaversMask/distilled-tiny-rnns-benchmark"
PINNED_COMMIT = "a8f6ae37c939ce8607f8751982b494c95da89889"
FOLLOWUP_REVISION_READY = True
if IN_COLAB and not FOLLOWUP_REVISION_READY:
    raise RuntimeError("This follow-up launcher is awaiting its verified runtime release. Use the completed reviewed notebook.")
if IN_COLAB:
    destination = Path(tempfile.mkdtemp(prefix="distilled-source-")) / "project"
    PROJECT = fetch_public_project(GITHUB_REPOSITORY, PINNED_COMMIT, destination)
    drive.mount("/content/drive")
    DRIVE_PROJECT = Path("/content/drive/MyDrive/distilled-tiny-rnns")
    RUNS = DRIVE_PROJECT / "runs"
else:
    PROJECT = Path(os.environ.get("DISTILLED_PROJECT_ROOT", Path.cwd())).resolve()
    if not (PROJECT / "pyproject.toml").exists() and (PROJECT.parent / "pyproject.toml").exists():
        PROJECT = PROJECT.parent
    RUNS = Path(os.environ.get("DISTILLED_OUTPUT_ROOT", str(PROJECT / "outputs/notebook")))

digest = hashlib.sha256()
for path in sorted((PROJECT / "src/distilled_rnns").glob("*.py")):
    digest.update(path.name.encode())
    digest.update(path.read_bytes())
assert digest.hexdigest() == EXPECTED_SOURCE_SHA256, "Downloaded source differs from the verified notebook revision."
os.chdir(PROJECT)
RUNS.mkdir(parents=True, exist_ok=True)
print("Project:", PROJECT)
print("Saved runs:", RUNS)


## Install and verify

The notebook checks the downloaded source, activates it immediately in this running kernel, and runs the verification suite before training. The GPU resume test runs on Colab; CPU verification alone cannot substitute for it. Exact package versions are recorded with each run.


In [ ]:
# A different notebook revision may already be imported in this kernel.
# Never mix its modules with this verified source download.
expected_package = (PROJECT / "src/distilled_rnns").resolve()
for module_name, module in list(sys.modules.items()):
    if module_name == "distilled_rnns" or module_name.startswith("distilled_rnns."):
        module_file = getattr(module, "__file__", None)
        if module_file is None or not Path(module_file).resolve().is_relative_to(expected_package):
            raise RuntimeError("Another project revision is loaded. Restart the Colab session, then run this notebook from its first cell.")
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", ".[test]"], check=True)
# Editable installs create a startup path file; this kernel is already running.
# Activate the verified source immediately in both Colab and local kernels.
import importlib
source_dir = str((PROJECT / "src").resolve())
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)
importlib.invalidate_caches()
from distilled_rnns.config import ExperimentConfig
print("Project import: ready")
import torch
print("PyTorch:", torch.__version__)
if IN_COLAB and not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime before continuing; this notebook will not silently run the Colab pilot on CPU.")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU (local verification only)")
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], check=True)


## Small follow-up smoke test

A tiny study exercises regularization selection, repeated seeds, saved results, and resume. This is a software check, not a scientific result.


In [ ]:
from distilled_rnns.followup import FollowupConfig, run_followup

SMOKE_RUN = RUNS / ("followup-smoke-" + EXPECTED_SOURCE_SHA256[:12])
smoke_config = FollowupConfig.from_file(PROJECT / "configs/followup_smoke.json")
smoke = run_followup(smoke_config, SMOKE_RUN, device=DEVICE)
assert smoke["status"] == "complete", smoke
smoke_resumed = run_followup(smoke_config, SMOKE_RUN, device=DEVICE)
assert smoke_resumed["groups"] == smoke["groups"], "Resumed results differ"
assert smoke_resumed["calibration"]["selected_l2"] == smoke["calibration"]["selected_l2"]
print("Follow-up smoke test and resume passed.")


## Run the bounded follow-up

**Calibration:** the original seed, 20261029, has already been inspected and is exploratory. Train four solo candidates with L2 strengths **0, 0.001, 0.01, and 0.1**. Each person's checkpoint is chosen using observed-choice validation NLL. Choose one strength using the mean validation NLL across all people; ties favor the smaller strength. This population-selected choice is dominated by the 98% typical majority; it is not selected specifically for rare people. The choice is saved before computing test metrics. Rare-group labels and true-policy scores do not choose the strength.

**Repeated runs:** freeze the selected strength for five new simulation seeds, **20261101, 20261111, 20261121, 20261131, 20261141** (integer random seeds, not dates). Compare the teacher, longer-trained unregularized solo students, calibrated solo students, and distilled students. If calibration selects zero, the calibrated and unregularized solo results are identical. The student arms start from the same weights within each seed. Population seeds are spaced to avoid reusing random initialization streams across populations. Training remains masked, and validation selects each person's checkpoint.

Calibration results stay separate from the repeated-seed summary. Keep the default protocol for this run: changing settings or code requires a new output folder. No test score should be used to choose settings for these same seeds.


In [ ]:
RUN_NAME = "followup-v1"  # Keep for resume; use a new name for changed settings or source.
CONFIG_NAME = "followup" if IN_COLAB else os.environ.get("DISTILLED_FOLLOWUP_CONFIG", "followup_smoke")
config = FollowupConfig.from_file(PROJECT / "configs" / f"{CONFIG_NAME}.json")
print(json.dumps(config.to_dict(), indent=2))
report = run_followup(config, RUNS / RUN_NAME, device=DEVICE)
if report["status"] != "complete":
    print(report["message"])


## Results and convergence checks

Inspect the validation curves and the fraction of people whose selected checkpoint is at the training limit. A boundary warning means further convergence checking is needed; it is not evidence for or against the research hypothesis. More training steps alone do not guarantee convergence.

NLL measures prediction of held-out choices and KL measures disagreement with the known generating policy; lower is better for both. Diagnostic scores have their support counts reported and can be unavailable. The summary reports descriptive differences across the five new seeds. It does not pool the calibration run into those results or establish statistical significance.

Review the comparison with both solo baselines. The same observed histories remain available as causal inputs, so these are conditional predictions rather than free-running strategy recovery. Cost conversion uses your supplied **1.07 units per T4-hour** only for recorded T4 training; Colab also bills setup and idle time.


In [ ]:
if report["status"] == "complete":
    from IPython.display import display, Image, Markdown
    print("Frozen solo L2 strength:", report["calibration"]["selected_l2"])
    print("Population-selected L2 is dominated by the typical majority (98% in the default study).")
    if report["calibration"]["selected_l2"] == 0:
        print("Zero was selected: calibrated solo and unregularized solo are the same model.")
    if report["calibration"].get("lambda_at_grid_boundary"):
        print("CHECK REQUIRED: the selected L2 strength is at a search-grid boundary.")
    for warning in report.get("convergence_warnings", []):
        print("CHECK REQUIRED:", warning)
    def _score_text(summary, paired=False):
        support = f"({summary['supported_seed_count']}/{summary['planned_seed_count']} seeds)"
        if summary["mean"] is None:
            return "unavailable " + support
        if paired:
            return f"{summary['mean']:+.4f} [{summary['min']:+.4f}, {summary['max']:+.4f}] {support}"
        return f"{summary['mean']:.4f} {support}"

    display(Markdown("**Scores:** seed means in nats per trial; lower is better. Parentheses show supported/planned seeds. "
                     "**Paired differences:** mean [minimum, maximum] across seeds. Positive NLL gain favors distillation; "
                     "positive KL loss means distillation has worse policy fidelity. These are descriptive summaries."))
    labels = {"teacher": "Teacher", "solo_unregularized": "Solo, no L2",
              "solo": "Calibrated solo", "distilled": "Distilled"}
    for group_name, group in report["groups"].items():
        rows = [f"**{group_name.title()} group**", "", "| Model | NLL mean | KL mean | Diagnostic NLL mean |",
                "| --- | ---: | ---: | ---: |"]
        for name in ("teacher", "solo_unregularized", "solo", "distilled"):
            metrics = group["models"][name]
            values = [_score_text(metrics[metric]) for metric in ("nll", "kl", "diagnostic_nll")]
            rows.append("| " + " | ".join([labels[name], *values]) + " |")
        rows.extend(["", "| Distilled versus | NLL gain | KL loss | Diagnostic NLL gain |",
                     "| --- | ---: | ---: | ---: |"])
        for baseline in ("solo", "solo_unregularized"):
            metrics = group["comparisons"][baseline]
            values = [_score_text(metrics[metric], paired=True)
                      for metric in ("likelihood_gain", "fidelity_loss", "diagnostic_likelihood_gain")]
            rows.append("| " + " | ".join([labels[baseline], *values]) + " |")
        support = [f"{row['seed']}: {row['diagnostic_count']} trials / {row['diagnostic_subject_count']} people"
                   for row in group["per_seed"]]
        rows.extend(["", "Diagnostic test support — " + "; ".join(support) + "."])
        display(Markdown("\n".join(rows)))
    timing = report["timing"]
    print(f"Recorded training time: {timing['cumulative_training_seconds'] / 60:.2f} minutes.")
    units = timing["t4_equivalent_units_at_user_rate"]
    if units is None:
        print("T4 compute-unit proxy unavailable for this device history.")
    else:
        print(f"Training-only T4 compute-unit proxy: {units:.4f} units at your supplied rate.")
    print("Includes calibration candidates and repeated fits; setup, evaluation and idle billing are additional.")
    output_dir = Path(report["output_dir"])
    plots = sorted(output_dir.rglob("convergence.png"))
    for plot in plots[:2]:
        print(plot.parent.name)
        display(Image(filename=str(plot)))
    for plot in plots[2:]:
        print("Additional convergence figure:", plot)
    print("Saved:", output_dir)
    print("Keep the entire followup-v1 folder, including summary.json, the frozen protocol/selection, every seed folder, and all checkpoints and environment records.")
else:
    print("Follow-up incomplete. Resume the same run before interpreting a repeated-seed summary.")
